In [1]:
import pandas as pd
import numpy as np
import scipy
from tqdm import tqdm

from bert_score import BERTScorer
from easse.bleu import corpus_bleu
from easse.fkgl import corpus_fkgl
from easse.samsa import get_samsa_sentence_scores
from easse.sari import corpus_sari
from easse.bertscore import corpus_bertscore


from utils import read_test_set, collect_references, sigmoid 

## Experimental Setting

Read the datasets with original sentences and references

In [2]:
hsplit_orig, hsplit_refs = read_test_set("hsplit_test", as_lists=True)
EVAL_DATASETS = { "hsplit": (hsplit_orig, hsplit_refs, 4)}

Same pre-processing parameters for all metrics

In [3]:
lowercase = False  # case-insensitive
tokenizer = "moses"

## Compute metrics for the Simplicity-DA dataset

In [4]:
df = pd.read_csv("data/Sulem-18.csv")

In [5]:
df.groupby("sys_name").count()  # 100 sentences per system

,sent_id,sys_type,orig_sent,simp_sent,grammaticality,meaning,simplicity,structural_simplicity
sys_name,,,,,,,,
DSS,70,70,70,70,70,70,70,70
DSS^m,70,70,70,70,70,70,70,70
Hybrid,70,70,70,70,70,70,70,70
Moses,70,70,70,70,70,70,70,70
NTS-h1_default_model,70,70,70,70,70,70,70,70
NTS-h1_w2v_model,70,70,70,70,70,70,70,70
NTS-h4_default_model,70,70,70,70,70,70,70,70
NTS-h4_w2v_model,70,70,70,70,70,70,70,70
SBMT-SARI,70,70,70,70,70,70,70,70


In [7]:
df_metrics_segment = pd.DataFrame()

In [ ]:
metrics = []
bertscore_rescale = BERTScorer(lang="en", rescale_with_baseline=True)
for _, row in tqdm(df.iterrows()):
    for test_set, (test_set_orig, test_set_refs, num_refs) in EVAL_DATASETS.items():
        orig_sents, ref_sents = collect_references(
            [row["sent_id"]], test_set_orig, test_set_refs, num_refs
        )
        
        # BLEU
        bleu_sys_refs = corpus_bleu(
            [row["simp_sent"]],
            ref_sents,
            smooth_method="floor",
            tokenizer=tokenizer,
            lowercase=lowercase,
            effective_order=True,
        )
            
        # SARI
        sari_score = corpus_sari(
            orig_sents,
            [row["simp_sent"]],
            ref_sents,
            tokenizer=tokenizer,
            lowercase=lowercase,
            use_f1_for_deletion=False,
        )
        
        
        # Flesch
        fkgl_sys = -corpus_fkgl([row["simp_sent"]], tokenizer=tokenizer)

        # BERTScore
        ref_sents = [ref for [ref] in ref_sents]
        bertscore_rescale_scores = bertscore_rescale.score([row["simp_sent"]], [ref_sents])
             
        metrics.append(
            {
                "sent_id": row["sent_id"],
                "sys_name": row["sys_name"],
                "test_set": test_set,
                "bleu": bleu_sys_refs,
                "sari": sari_score,
                "fkgl": fkgl_sys,
                "bertscore_F1": bertscore_rescale_scores[2].cpu().item(),
            }
        )

df_metrics_segment = pd.DataFrame(metrics)
df_metrics_segment.to_csv('data/metrics_results.csv')

Some weights of the model checkpoint at roberta-large were not used when initializing RobertaModel: ['lm_head.dense.bias', 'lm_head.decoder.weight', 'lm_head.bias', 'lm_head.dense.weight', 'lm_head.layer_norm.weight', 'lm_head.layer_norm.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
1750it [08:10,  3.57it/s]


In [9]:
import math
from collections import Counter

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# -------------------------------------------------------------------
# 1. Load LM for PPL / SLOR
# -------------------------------------------------------------------
lm_name = "gpt2"
tokenizer = GPT2TokenizerFast.from_pretrained(lm_name)
model = GPT2LMHeadModel.from_pretrained(lm_name)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# GPT-2 has no pad_token by default; set it for batching safety if needed
#if tokenizer.pad_token is None:
tokenizer.pad_token = tokenizer.eos_token

# -------------------------------------------------------------------
# 2. Build unigram distribution over ALL simplified sentences
#    (for SLOR baseline; you can replace with a larger corpus if you want)
# -------------------------------------------------------------------
unigram_counts = Counter()
total_tokens = 0

for simp in df["simp_sent"]:
    # tokenize at LM-level (subword tokens)
    toks = tokenizer.tokenize(simp)
    unigram_counts.update(toks)
    total_tokens += len(toks)

V = len(unigram_counts)
alpha = 1.0  # Laplace smoothing


def unigram_logprob(text: str):
    """Return (log P_uni(text), N_tokens) under unigram LM."""
    toks = tokenizer.tokenize(text)
    logp = 0.0
    for t in toks:
        c = unigram_counts.get(t, 0)
        p = (c + alpha) / (total_tokens + alpha * V)
        logp += math.log(p + 1e-12)
    return logp, len(toks)


# -------------------------------------------------------------------
# 3. Helper: sentence LM log-prob + PPL
# -------------------------------------------------------------------
@torch.no_grad()
def lm_logprob_and_ppl(text: str) :
    """
    Returns (log P_LM(text), PPL, N_tokens) for GPT-2.
    log probability is sum over tokens, PPL uses natural base.
    """
    enc = tokenizer(text, return_tensors="pt")
    input_ids = enc["input_ids"].to(device)

    # GPT-2: loss is mean cross-entropy over tokens
    out = model(input_ids, labels=input_ids)
    ce = float(out.loss.item())              # mean NLL per token (in nats)
    N = input_ids.size(1)
    logp_lm = -ce * N                        # sum log-probability
    ppl = math.exp(ce)                       # perplexity = exp(average NLL)
    return logp_lm, ppl, N


In [ ]:

PPL_score = []   # new
SLOR_score = []  # new

for comp, simp in zip(df["orig_sent"],
                      df["simp_sent"]):

    

    # ---- LM-based metrics on the simplified sentence ----
    logp_lm, ppl, N = lm_logprob_and_ppl(simp)
    logp_uni, N_uni = unigram_logprob(simp)
    # use the same N for safety; if either is 0, SLOR is undefined → set to 0
    N_eff = max(N, N_uni, 1)

    slor = (logp_lm - logp_uni) / N_eff

    PPL_score.append(ppl)
    SLOR_score.append(slor)

# -------------------------------------------------------------------
# 5. Attach to df_metrics_segment (same replication pattern you use)
# -------------------------------------------------------------------
k = len(EVAL_DATASETS)  # how many datasets you’re repeating over

# NEW: add LM-based metrics
df_metrics_segment["PPL"]       = [p  for p  in PPL_score  for _ in range(k)]
df_metrics_segment["SLOR"]      = [s  for s  in SLOR_score for _ in range(k)]
df_metrics_segment.to_csv('data/metrics_results.csv', index=False)

### Compute MREF

In [ ]:

import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
from MREF import MREF,AxisWeights
import pandas as pd
import numpy as np
import scipy
from tqdm import tqdm
from utils import read_test_set

hsplit_orig, hsplit_refs = read_test_set("hsplit_test", as_lists=True)
EVAL_DATASETS = { "hsplit": (hsplit_orig, hsplit_refs, 4)}
df = pd.read_csv("data/Sulem-18.csv")
lowercase = False  # case-insensitive
tokenizer = "moses"
df_metrics_segment = pd.read_csv("data/metrics_results.csv")



mref = MREF(
        language="en",
       # axis_weights=AxisWeights(w_G=1.0, w_M=1.0, w_S=1.0)
    )


S_axis=[]
M_axis=[]
G_axis=[]
Mref_score=[]


for comp,simp in zip(df["orig_sent"],df["simp_sent"]):

    
    scores = mref.score_axes(comp, simp)
    
    S_axis.append(scores["S_axis"])
    G_axis.append(scores["G_axis"])
    M_axis.append(scores["M_axis"])
    Mref_score.append(scores["MREFscore"])
    
    #print(len(S_score))
    #print(f'{comp} {simp} : {SS}, {GS}, {MS}, {CES}')
df_metrics_segment["S_axis"] = [ s for s in S_axis for _ in range(len(EVAL_DATASETS))]
df_metrics_segment["M_axis"] = [ m for m in M_axis for _ in range(len(EVAL_DATASETS))]
df_metrics_segment["G_axis"] = [ g for g in G_axis for _ in range(len(EVAL_DATASETS))]
df_metrics_segment["Mref_score"] = [ ce for ce in Mref_score for _ in range(len(EVAL_DATASETS))]
df_metrics_segment.to_csv('data/metrics_results.csv', index=False)

## Compute QuestEval

In [ ]:
from questeval.questeval_metric import QuestEval
#import tqdm
import sys, questeval, transformers, huggingface_hub

questeval = QuestEval(no_cuda=True)
"""
data: list of (TC, TS, y_human)
returns: list of QuestEval F1 scores (or dict of P/R/F1 if you prefer)
"""
questeval_score=[]
for src_texts,hyp_texts in zip(df["orig_sent"],df["simp_sent"]):
  
    score = questeval.corpus_questeval(hypothesis=[hyp_texts], sources=[src_texts])
    questeval_score.append(score['ex_level_scores'][0])
    
df_metrics_segment['QuestEval']= [ r for r in questeval_score for _ in range(len(EVAL_DATASETS))]
df_metrics_segment.to_csv('data/metrics_results.csv', index=False)


Using the latest cached version of the module from C:\Users\engmo\.cache\huggingface\modules\datasets_modules\metrics\bertscore\acd7f806e3c6996af65006355eeb46c0a8a6ac0009344c2f3224f66d483cf70a (last modified on Sat Oct 25 09:05:48 2025) since it couldn't be found locally at bertscore\bertscore.py or remotely (ConnectionError).


## Compute SLE 

In [ ]:
import torch
from sle.scorer import SLEScorer
device = "cuda" if torch.cuda.is_available() else "cpu"
#scorer = SLEScorer("liamcripwell/sle-base")
scorer = SLEScorer("F:\models\sle-base",device=device)

sle=[]
 
results = scorer.score(df["simp_sent"],inputs=df["orig_sent"])
 

df_metrics_segment["sle"] = [ s for s in results['sle'] for _ in range(len(EVAL_DATASETS))]
df_metrics_segment.to_csv('data/metrics_results.csv', index=False)


100%|██████████| 219/219 [00:59<00:00,  3.67it/s]


### Compute SAMSA

In [ ]:
# Now compute SAMSA scores and add to the dataframe
#should change the invironment to that compatible with samsa
import pandas as pd
import numpy as np
import scipy
from tqdm import tqdm

from easse.samsa import get_samsa_sentence_scores
from utils import read_test_set

hsplit_orig, hsplit_refs = read_test_set("hsplit_test", as_lists=True)
EVAL_DATASETS = { "hsplit": (hsplit_orig, hsplit_refs, 4)}
df = pd.read_csv("data/Sulem-18.csv")
lowercase = False  # case-insensitive
tokenizer = "moses"
df_metrics_segment = pd.read_csv("metrics_results_without_samsa.csv")
#import spacy
#spacy.load('en_core_web_md')

samsa_scores = get_samsa_sentence_scores(
    df["orig_sent"],
    df["simp_sent"],
    tokenizer=tokenizer,
    lowercase=lowercase,
)

# Since SAMSA is reference-less, this reformating is only done so that it can appear in thae same dataframe as the other metrics
df_metrics_segment["samsa"] = [
    s for s in samsa_scores for _ in range(len(EVAL_DATASETS))
]
df_metrics_segment.to_csv('data/metrics_results.csv', index=False)

Loading spaCy model 'en_core_web_md'... Done (10.807s).


[dynet] 2.1


Loading from 'C:\Users\engmo\easse\easse\resources\tools\ucca-bilstm-1.3.10\models\ucca-bilstm.json'.
Loading from 'C:\Users\engmo\easse\easse\resources\tools\ucca-bilstm-1.3.10\models\ucca-bilstm.enum'... Done (0.034s).
Loading model from 'C:\Users\engmo\easse\easse\resources\tools\ucca-bilstm-1.3.10\models\ucca-bilstm': 23param [00:44,  1.93s/param]                    
Loading model from 'C:\Users\engmo\easse\easse\resources\tools\ucca-bilstm-1.3.10\models\ucca-bilstm': 100%|██████████| 23/23 [00:43<00:00,  1.90s/param]
Loading from 'C:\Users\engmo\easse\easse\resources\tools\ucca-bilstm-1.3.10\models\ucca-bilstm.nlp.json'.
tupa --hyperparams "shared --lstm-layers 2" "amr --max-edge-labels 110 --node-label-dim 20 --max-node-labels 1000 --node-category-dim 5 --max-node-categories 25" "sdp --max-edge-labels 70" "conllu --max-edge-labels 60" --log parse.log --max-words 0 --max-words-external 249861 --vocab C:\Users\engmo\easse\easse\resources\tools\ucca-bilstm-1.3.10\vocab\en_core_web_l